# Sheet 02: Data Cleaning & Feature Engineering

Handle missing values, remove any duplicates, and engineer the target
variable `resistance_level` (Low / Medium / High) used for classification.


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/ears_net.csv")
print("Loaded:", df.shape)
df.head()

Loaded: (3628, 6)


,country,year,bacterium,antibiotic,n_isolates,pct_resistant
0,Austria,2012,Acinetobacter spp.,Aminoglycosides,NaN,NaN
1,Austria,2013,Acinetobacter spp.,Aminoglycosides,51.0,9.80000
2,Austria,2014,Acinetobacter spp.,Aminoglycosides,79.0,8.86076
3,Belgium,2012,Acinetobacter spp.,Aminoglycosides,NaN,NaN
4,Belgium,2013,Acinetobacter spp.,Aminoglycosides,3.0,NaN


In [3]:
df.isnull().sum()

country            0
year               0
bacterium          0
antibiotic         0
n_isolates       110
pct_resistant    206
dtype: int64

In [4]:
before = len(df)
df = df.dropna(subset=['pct_resistant'])
after = len(df)
print(f"Dropped {before - after} rows with missing pct_resistant.")
print("Remaining:", df.shape)

# check what missing values are left
df.isnull().sum()

Dropped 206 rows with missing pct_resistant.
Remaining: (3422, 6)


country          0
year             0
bacterium        0
antibiotic       0
n_isolates       0
pct_resistant    0
dtype: int64

In [5]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


## Feature Engineering: Creating the Target Variable

The model predicts a **category**, so I convert the continuous `pct_resistant`
into three public-health-meaningful bands:

- **Low**: resistance < 10%
- **Medium**: 10% – 25%
- **High**: ≥ 25%

These thresholds reflect standard surveillance concern levels and align roughly
with the data's quartiles, giving interpretable, reasonably balanced classes.
This `resistance_level` column is the target (`y`) for classification.

In [6]:
def classify_resistance(pct):
    if pct < 10:
        return "Low"
    elif pct < 25:
        return "Medium"
    else:
        return "High"

# Apply the function to every row's pct_resistant value
df['resistance_level'] = df['pct_resistant'].apply(classify_resistance)

df[['country', 'year', 'bacterium', 'antibiotic', 'pct_resistant', 'resistance_level']].head(10)

,country,year,bacterium,antibiotic,pct_resistant,resistance_level
1,Austria,2013,Acinetobacter spp.,Aminoglycosides,9.80000,Low
2,Austria,2014,Acinetobacter spp.,Aminoglycosides,8.86076,Low
6,Bulgaria,2012,Acinetobacter spp.,Aminoglycosides,58.50000,High
7,Bulgaria,2013,Acinetobacter spp.,Aminoglycosides,58.20000,High
8,Bulgaria,2014,Acinetobacter spp.,Aminoglycosides,63.63636,High
10,Croatia,2013,Acinetobacter spp.,Aminoglycosides,92.10000,High
11,Croatia,2014,Acinetobacter spp.,Aminoglycosides,88.02395,High
12,Cyprus,2012,Acinetobacter spp.,Aminoglycosides,52.20000,High
13,Cyprus,2013,Acinetobacter spp.,Aminoglycosides,60.60000,High
14,Cyprus,2014,Acinetobacter spp.,Aminoglycosides,73.68421,High


In [7]:
df['resistance_level'].value_counts()

resistance_level
Low       1425
High      1030
Medium     967
Name: count, dtype: int64

In [8]:
df = df.reset_index(drop=True)   # renumber rows cleanly 0,1,2,...
print("Final shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
df.head()

Final shape: (3422, 7)
Missing values: 0
Duplicates: 0


,country,year,bacterium,antibiotic,n_isolates,pct_resistant,resistance_level
0,Austria,2013,Acinetobacter spp.,Aminoglycosides,51.0,9.80000,Low
1,Austria,2014,Acinetobacter spp.,Aminoglycosides,79.0,8.86076,Low
2,Bulgaria,2012,Acinetobacter spp.,Aminoglycosides,65.0,58.50000,High
3,Bulgaria,2013,Acinetobacter spp.,Aminoglycosides,91.0,58.20000,High
4,Bulgaria,2014,Acinetobacter spp.,Aminoglycosides,99.0,63.63636,High


In [9]:
df.to_csv("../data/processed/ears_net_clean.csv", index=False)
print("Saved cleaned dataset with resistance_level target.")


Saved cleaned dataset with resistance_level target.
